#  Step 1: Load the data

In [1]:
from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np

drive.mount('/content/drive')

DRIVE     = Path('/content/drive/MyDrive/m5_project')
PROCESSED = DRIVE / 'processed'

# split boundaries (day numbers), same as the paper's M5 setup
TRAIN_END  = 1885      # last training day
VAL_START  = 1886      # validation: d_1886 - d_1913
TEST_START = 1914      # test: d_1914 - d_1941 (scored once, at the very end)
HORIZON    = 28

df = pd.read_parquet(PROCESSED / 'ca12_long.parquet',
                     columns=['id', 'item_id', 'dept_id', 'cat_id', 'store_id',
                              'd', 'sales', 'sell_price'])
df['sales'] = df['sales'].astype('int32')     # int16 would overflow when we add up levels

print(df.shape)                                # expect (6096944, 8)
print('days:', df.d.min(), '->', df.d.max())   # expect 847 -> 1941

Mounted at /content/drive
(6096944, 8)
days: 847 -> 1941


# Step 2: Define 8 levels

- Defines how item-store sales are added up into higher levels (the "tree").

In [4]:
LEVELS = {
    'L1_total':      [],
    'L2_store':      ['store_id'],
    'L3_cat':        ['cat_id'],
    'L4_dept':       ['dept_id'],
    'L5_store_cat':  ['store_id', 'cat_id'],
    'L6_store_dept': ['store_id', 'dept_id'],
    'L7_item':       ['item_id'],
    'L8_item_store': ['item_id', 'store_id'],
}

# teacher levels for H-Pro-Avg: (L1-L4)
for name, cols in LEVELS.items():
    n = 1 if not cols else df[cols].drop_duplicates().shape[0]
    print(f'{name:<15} {n:>5} series')

L1_total            1 series
L2_store            2 series
L3_cat              3 series
L4_dept             7 series
L5_store_cat        6 series
L6_store_dept      14 series
L7_item          3049 series
L8_item_store    6098 series



| Level | Count | Meaning |
|---|---|---|
| L1 Total | 1 | both stores, all products added together |
| L2 Store | 2 | one series per store (CA_1, CA_2) |
| L3 Category | 3 | FOODS, HOBBIES, HOUSEHOLD |
| L4 Department | 7 | e.g. FOODS_1, FOODS_2, FOODS_3 ... |
| L5 Store × Category | 6 | 2 stores × 3 categories |
| L6 Store × Department | 14 | 2 stores × 7 departments |
| L7 Item | 3,049 | each product, both stores added |
| L8 Item × Store | 6,098 | each product in each store (bottom level) |

# Step 3: The bottom table (one row per product-store)
- Right now the data is long (one row per product per day).
- To add things up quickly, we turn it into a grid: one row per product-store and one column per day. Every level is built from this grid.

In [5]:
# grid: rows = 6,098 product-stores, columns = days, values = units sold
bottom = df.pivot(index='id', columns='d', values='sales').fillna(0).astype('int32')

# for each product-store: its item, dept, category and store
meta = (df[['id', 'item_id', 'dept_id', 'cat_id', 'store_id']].drop_duplicates('id').set_index('id').astype(str))
meta = meta.loc[bottom.index]
bottom.index = bottom.index.astype(str)
print(bottom.shape)                    # expect (6098, 1095)
assert bottom.values.sum() == df.sales.sum(), 'units lost while building grid'

(6098, 1095)


# Step 4: Add up every level
- This builds the tree. Each level’s daily sales = the bottom rows added up by that level’s columns.

In [6]:
def aggregate(level_cols):
    """Add up bottom rows by the given columns ([] = everything)."""
    if not level_cols:
        return bottom.sum().to_frame('Total').T                  # L1: one row, all units
    return bottom.join(meta[level_cols]).groupby(level_cols).sum()

# build all 8 levels
sales_by_level = {name: aggregate(cols) for name, cols in LEVELS.items()}

# check: every level must add up to the same grand total
grand_total = bottom.values.sum()
for name, table in sales_by_level.items():
    assert table.values.sum() == grand_total, f'{name} does not add up'
    print(f'{name:<15} {table.shape[0]:>5} series x {table.shape[1]} days')

sales_by_level['L2_store'].iloc[:, :5]       # peek: first 5 days for each store

L1_total            1 series x 1095 days
L2_store            2 series x 1095 days
L3_cat              3 series x 1095 days
L4_dept             7 series x 1095 days
L5_store_cat        6 series x 1095 days
L6_store_dept      14 series x 1095 days
L7_item          3049 series x 1095 days
L8_item_store    6098 series x 1095 days


,847,848,849,850,851
store_id,,,,,
CA_1,4390,4766,4416,4390,3575
CA_2,3355,3981,3580,3807,2691
